# Camera / PhotogrammetryConfig — full method smoke test

This notebook exercises **every method** on `PhotogrammetryConfig` and `Camera` from `camera.py`.

Many methods require real data (video files, Alvium logs, a populated `IPA_flight` install, a `Flight` object from `pils`, etc.) that we can't fabricate here. Each cell:

- Runs the real call when the required inputs are present, or
- Falls back to a clearly-labelled mock / direct-call so the method signature and code path still execute.

**Before running:** edit the `CONFIG` block below with real paths on your machine.

In [27]:
import sys
sys.path.insert(0, ".")  # adjust so `camera.py` is importable

from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd

sys.path.append("/home/fastori/Desktop/POLOCALC/ARS/pils")
sys.path.append("/home/fastori/Desktop/POLOCALC/ARS/")

from pils.sensors.camera import PhotogrammetryConfig, Camera


## 0. Configuration — edit these paths

In [7]:
CONFIG = {
    # A YAML config with a "<model>_camera" block + a "pipeline" block
    "config_yaml": "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/config/photogrammetryConfig.yaml",
    "camera_model": "sony",  # or "alvium"

    # Sony: a folder containing one or more .mp4 files, with a sibling *.log
    "sony_dir": "/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera",

    # Alvium: a folder containing the *.log, with images under ../../proc/images/
    "alvium_dir": None,

    # Photogrammetry-results mode: folder or path to a CSV
    "photogrammetry_csv": "/home/fastori/Desktop/POLOCALC/ARS/IPA_flight_no_git/output_202512/20251206/output_FLY20251206153000/attitude_reconstruction_FLY20251206_153000.parquet",

    # CSV of geodetic targets used by run_photogrammetry / run_photogrammetry_multi_flights
    "targets_csv": "/data/POLOCALC/campaigns/202511/metadata/202511_coordinates.csv",

    "output_dir": "outputs/photogrammetry_test",
}


## 1. `PhotogrammetryConfig`

In [8]:
# PhotogrammetryConfig.__init__  (internally calls _parse -> _resolve_camera_model)
try:
    pg_config = PhotogrammetryConfig(CONFIG["config_yaml"], camera_model=CONFIG["camera_model"])
    print(pg_config)  # __repr__
except FileNotFoundError as e:
    print("Config not found — using a stub object for the rest of the notebook:", e)

    class _StubConfig:
        camera_matrix = np.eye(3)
        distortion_coeffs = np.zeros(5)
        reference_point = np.array([0.0, 0.0, 0.0])
        dji_base_logged = None
        finder = {}
        pnp = {}
        mcmc = {}
        drone_correlation = {}
        telescope = {}
        polarization = {}
        def __repr__(self):
            return "StubConfig()"

    pg_config = _StubConfig()
    print(pg_config)


PhotogrammetryConfig(config=/home/fastori/Desktop/POLOCALC/ARS/pils/pils/config/photogrammetryConfig.yaml, camera_matrix=(3, 3))


In [9]:
# PhotogrammetryConfig._resolve_camera_model (private) — call directly to see resolution logic
if isinstance(pg_config, PhotogrammetryConfig):
    print(pg_config._resolve_camera_model())
else:
    print("Skipped: pg_config is the stub, not a real PhotogrammetryConfig instance.")


sony_camera


## 2. `Camera.__init__` — one instance per mode

In [ ]:
# Mode 1: photogrammetry results mode
cam_pg = Camera(CONFIG["photogrammetry_csv"], use_photogrammetry=True)
print(cam_pg)  # __repr__ before load_data (data is None)

# Mode 2: Sony video mode
if CONFIG["sony_dir"] is not None:
    cam_sony = Camera(CONFIG["sony_dir"], use_photogrammetry=False)
    print(cam_sony)
else:
    cam_sony = None

# Mode 3: Alvium image-sequence mode
if CONFIG["alvium_dir"] is not None:
    cam_alvium = Camera(CONFIG["alvium_dir"], use_photogrammetry=False)
    print(cam_alvium)
else:
    cam_alvium = None


Camera(path=/home/fastori/Desktop/POLOCALC/ARS/IPA_flight_no_git/output_202512/20251206/output_FLY20251206153000/attitude_reconstruction_FLY20251206_153000.parquet, mode=video, model=not loaded)
Camera(path=/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera, mode=video, model=not loaded)


## 3. `Camera.load_data` (dispatches to the private `_load_*` methods)

In [15]:
for name, cam in [("photogrammetry", cam_pg), ("sony", cam_sony), ("alvium", cam_alvium)]:
    try:
        cam.load_data()
        print(f"[{name}] loaded OK -> {cam}")
    except FileNotFoundError as e:
        print(f"[{name}] skipped (no data on disk): {e}")
    except Exception as e:
        print(f"[{name}] failed: {e}")


2026-07-01 17:54:54,397 - pils.sensors.camera - INFO - Loading photogrammetry data from /home/fastori/Desktop/POLOCALC/ARS/IPA_flight_no_git/output_202512/20251206/output_FLY20251206153000/attitude_reconstruction_FLY20251206_153000.parquet
2026-07-01 17:54:54,610 - pils.sensors.camera - INFO - Parsing telemetry from /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (5040.2 MB, timeout=2520s)


[photogrammetry] failed: invalid utf-8 sequence
   ⏳ telemetry_parser running... 0s / 2520s


Process Process-1:
Traceback (most recent call last):
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/sensors/camera.py", line 530, in _worker
    imu_data = parser.normalized_imu()
               ^^^^^^^^^^^^^^^^^^^^^^^
pyo3_runtime.PanicException: assertion failed: v.len() == 3
thread '<unnamed>' panicked at /home/runner/work/telemetry-parser/telemetry-parser/src/sony/mod.rs:41:9:
assertion failed: v.len() == 3
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace
2026-07-01 17:55:04,048 - pils.sensors.camera - WARNING - Sony telemetry unavailable for /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (telemetry-parser crashed (exitcode=

[sony] loaded OK -> Camera(path=/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera, mode=video, model=sony)
[alvium] failed: 'NoneType' object has no attribute 'load_data'


In [16]:
# Direct calls to the private loaders (useful for isolated testing / debugging)

# Camera._load_photogrammetry_data
try:
    df, model = cam_pg._load_photogrammetry_data()
    print("photogrammetry data:", df.shape, model)
except Exception as e:
    print("skipped _load_photogrammetry_data:", e)

# Camera._load_sony_camera_data (needs a list of .mp4 Paths)
try:
    video_files = [p for p in Path(CONFIG["sony_dir"]).iterdir() if p.suffix.lower() == ".mp4"]
    df, model = cam_sony._load_sony_camera_data(video_files)
    print("sony data:", df.shape, model)
except Exception as e:
    print("skipped _load_sony_camera_data:", e)

# Camera._load_alvium_camera_data
try:
    df, model = cam_alvium._load_alvium_camera_data()
    print("alvium data:", df.shape, model)
except Exception as e:
    print("skipped _load_alvium_camera_data:", e)


2026-07-01 17:55:15,002 - pils.sensors.camera - INFO - Loading photogrammetry data from /home/fastori/Desktop/POLOCALC/ARS/IPA_flight_no_git/output_202512/20251206/output_FLY20251206153000/attitude_reconstruction_FLY20251206_153000.parquet
2026-07-01 17:55:15,109 - pils.sensors.camera - INFO - Parsing telemetry from /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (5040.2 MB, timeout=2520s)


skipped _load_photogrammetry_data: invalid utf-8 sequence
   ⏳ telemetry_parser running... 0s / 2520s


Process Process-2:
Traceback (most recent call last):
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/sensors/camera.py", line 530, in _worker
    imu_data = parser.normalized_imu()
               ^^^^^^^^^^^^^^^^^^^^^^^
pyo3_runtime.PanicException: assertion failed: v.len() == 3
thread '<unnamed>' panicked at /home/runner/work/telemetry-parser/telemetry-parser/src/sony/mod.rs:41:9:
assertion failed: v.len() == 3
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace
2026-07-01 17:55:15,684 - pils.sensors.camera - WARNING - Sony telemetry unavailable for /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (telemetry-parser crashed (exitcode=

sony data: (0, 0) sony
skipped _load_alvium_camera_data: 'NoneType' object has no attribute '_load_alvium_camera_data'


## 4. Alvium-specific helpers

In [ ]:
# Camera._read_alvium_tstart — populates cam_alvium.tstart from the log file
try:
    cam_alvium.logpath = next(Path(CONFIG["alvium_dir"]).glob("*.[Ll][Oo][Gg]"))
    cam_alvium._read_alvium_tstart()
    print("tstart:", cam_alvium.tstart)
except StopIteration:
    print("skipped: no .log file found in alvium_dir")
except Exception as e:
    print("skipped _read_alvium_tstart:", e)


In [ ]:
# Camera._parse_alvium_log
try:
    log_df = cam_alvium._parse_alvium_log(cam_alvium.logpath)
    print(log_df)
except Exception as e:
    print("skipped _parse_alvium_log:", e)


## 5. Sony telemetry parsing (subprocess-isolated)

In [17]:
# Camera._parse_sony_telemetry — runs telemetry_parser + Madgwick filter in a subprocess.
# This can take a while and needs the `telemetry_parser` package + a real .mp4 with IMU metadata.
try:
    video_files = [p for p in Path(CONFIG["sony_dir"]).iterdir() if p.suffix.lower() == ".mp4"]
    telemetry_df = cam_sony._parse_sony_telemetry(str(video_files[0]))
    print(telemetry_df.head())
except Exception as e:
    print("skipped _parse_sony_telemetry:", e)


2026-07-01 17:55:31,395 - pils.sensors.camera - INFO - Parsing telemetry from /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (5040.2 MB, timeout=2520s)


   ⏳ telemetry_parser running... 0s / 2520s


Process Process-3:
Traceback (most recent call last):
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/sensors/camera.py", line 530, in _worker
    imu_data = parser.normalized_imu()
               ^^^^^^^^^^^^^^^^^^^^^^^
pyo3_runtime.PanicException: assertion failed: v.len() == 3
thread '<unnamed>' panicked at /home/runner/work/telemetry-parser/telemetry-parser/src/sony/mod.rs:41:9:
assertion failed: v.len() == 3
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace


skipped _parse_sony_telemetry: telemetry-parser crashed (exitcode=1) on /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4


## 6. Streaming / frame access

In [36]:
# --- Sony camera: confirm state ---
print(cam_sony)
print("fps:", cam_sony.fps)
print("tstart:", cam_sony.tstart)
print("logpath:", cam_sony.logpath)
print("is_image_sequence:", cam_sony.is_image_sequence)  # should be False for Sony
print("data shape:", cam_sony.data[0].shape if cam_sony.data else None)

Camera(path=/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera, mode=video, model=sony)
fps: 29.97002997002997
tstart: 2025-12-06 15:29:26.290000
logpath: /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/20251206_153038_file.log
is_image_sequence: False
data shape: (0, 0)


In [33]:
# Camera._reader_loop is started automatically as a background thread inside
# _load_sony_camera_data(); we don't call it directly, we just drain the queue it fills.

# Camera.get_next_frame — works for both video (queue-based) and image-sequence modes
try:
    result = cam_sony.get_next_frame()
    if result is not None:
        idx, frame = result
        print("next frame:", idx, frame.shape)
    else:
        print("get_next_frame returned None (exhausted or no data)")
except Exception as e:
    print("skipped get_next_frame:", e)


next frame: 0 (2160, 3840, 3)


In [34]:
# Camera.get_frame — random access, image-sequence mode only
try:
    frame0 = cam_sony.get_frame(1000)
    print("frame 0 shape:", frame0.shape)
except Exception as e:
    print("skipped get_frame:", e)


skipped get_frame: Random access is disabled for videos (streaming mode).


In [18]:
# Camera.get_timestamp
try:
    ts = cam_sony.get_timestamp(0)
    print("timestamp for frame 0:", ts)
except Exception as e:
    print("skipped get_timestamp:", e)


timestamp for frame 0: 2025-12-06 15:29:26.290000


## 7. Visualisation

In [19]:
# Camera.plot_frame — saves to disk instead of showing inline, so it works headlessly
try:
    Path("outputs").mkdir(exist_ok=True)
    cam_alvium.plot_frame(frame_number=0, color="rgb", save_path="outputs/frame0.jpg")
    print("saved outputs/frame0.jpg")
except Exception as e:
    print("skipped plot_frame:", e)


skipped plot_frame: 'NoneType' object has no attribute 'plot_frame'


## 8. Cleanup

In [ ]:
# Camera.release
for name, cam in [("photogrammetry", cam_pg), ("sony", cam_sony), ("alvium", cam_alvium)]:
    cam.release()
    print(f"[{name}] released, stopped={cam.stopped}")


## 9. Static helpers (`_build_dh_wrapper`, `_ensure_polars`)

In [20]:
# Camera._ensure_polars — converts pandas -> polars, passes polars through unchanged
pdf = pd.DataFrame({"a": [1, 2, 3]})
print(type(Camera._ensure_polars(pdf)))          # polars.DataFrame
print(type(Camera._ensure_polars(pl.DataFrame(pdf))))  # unchanged polars.DataFrame


<class 'polars.dataframe.frame.DataFrame'>
<class 'polars.dataframe.frame.DataFrame'>


In [21]:
# Camera._build_dh_wrapper — needs a `flight` object (from pils) with .raw_data / .flight_info.
# We fabricate a minimal stand-in so the wrapping/normalisation logic still runs.
from types import SimpleNamespace

drone_df = pd.DataFrame({
    "timestamp": [1_700_000_000, 1_700_000_001, 1_700_000_002],
    "lat": [45.1, 45.1, 45.1],
    "lon": [9.2, 9.2, 9.2],
})
litchi_series = pd.Series([1, 2, 3])

stub_flight = SimpleNamespace(
    raw_data=SimpleNamespace(
        drone_data=SimpleNamespace(drone=drone_df, litchi=litchi_series),
        payload_data=None,
    ),
    flight_info={"drone_data_folder_path": "data/flight01/drone"},
)

try:
    dh_wrapper = Camera._build_dh_wrapper(stub_flight, cam_alvium)
    print(dh_wrapper.camera)
    print(dh_wrapper.raw_data.drone_data.drone.dtypes)
    print(dh_wrapper.flight_info)
except Exception as e:
    print("skipped _build_dh_wrapper:", e)


None
timestamp             int64
lat                 float64
lon                 float64
datetime     datetime64[ns]
dtype: object
{'drone_data_folder_path': 'data/flight01/drone'}


## 10. Full pipeline entry points

`run_photogrammetry` and `run_photogrammetry_multi_flights` depend on the external
`IPA_flight` package plus a real `pils.Flight` object with drone/camera data attached,
so these calls will only succeed in a fully wired environment. They're included for
completeness — expect a clean, informative exception if `IPA_flight` or the flight
data isn't available.

In [28]:
# Camera.run_photogrammetry — single flight
try:
    result_df = cam_sony.run_photogrammetry(
        csv_file=CONFIG["targets_csv"],
        config=pg_config,
        flight=stub_flight,          # replace with a real pils.Flight in practice
        output_dir=CONFIG["output_dir"],
    )
    print(result_df.shape)
except Exception as e:
    print("skipped run_photogrammetry:", e)


skipped run_photogrammetry: 'types.SimpleNamespace' object has no attribute 'metadata'


In [29]:
# Camera.run_photogrammetry_multi_flights — static method, multiple flights at once
try:
    multi_df = Camera.run_photogrammetry_multi_flights(
        flights=[stub_flight, stub_flight],   # replace with real pils.Flight objects
        csv_file=CONFIG["targets_csv"],
        config=pg_config,
        output_dir=CONFIG["output_dir"],
    )
    print(multi_df.shape)
except Exception as e:
    print("skipped run_photogrammetry_multi_flights:", e)



  Preparing flight 1/2
[SKIP] Flight 1: drone data failed: 'types.SimpleNamespace' object has no attribute 'add_drone_data'

  Preparing flight 2/2
[SKIP] Flight 2: drone data failed: 'types.SimpleNamespace' object has no attribute 'add_drone_data'
No valid flights could be prepared — aborting.
(0, 0)


## 11. `_run_check_results` (diagnostic plotting, private)

In [30]:
# Needs the plotting callables from IPA_flight.utils plus a dict with the expected columns.
try:
    from IPA_flight.IPA_flight.utils import (
        draw_results_on_last_frame,
        plot_targets_coordinates,
        plot_attitude,
        plot_drone_gps_alignment,
        plot_telescope_pointing,
        plot_telescope_attitude,
    )

    dummy_dict = pl.DataFrame({
        "frame": [0, 1],
        "x": [10.0, 12.0],
        "y": [20.0, 22.0],
    })

    cam_alvium._run_check_results(
        dummy_dict,
        "demo_step",
        Path("outputs/plots"),
        draw_results_on_last_frame,
        plot_targets_coordinates,
        plot_attitude,
        plot_drone_gps_alignment,
        plot_telescope_pointing,
        plot_telescope_attitude,
    )
    print("check-results plots written to outputs/plots/")
except Exception as e:
    print("skipped _run_check_results:", e)


skipped _run_check_results: 'NoneType' object has no attribute '_run_check_results'


## 12. `__repr__`

In [31]:
for name, cam in [("photogrammetry", cam_pg), ("sony", cam_sony), ("alvium", cam_alvium)]:
    print(f"[{name}] {cam!r}")


[photogrammetry] Camera(path=/home/fastori/Desktop/POLOCALC/ARS/IPA_flight_no_git/output_202512/20251206/output_FLY20251206153000/attitude_reconstruction_FLY20251206_153000.parquet, mode=video, model=not loaded)
[sony] Camera(path=/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera, mode=video, model=sony)
[alvium] None
